# Predicción de Demanda por Categoría de Ataúd — Random Forest
## Funeraria Aranzabal

Pasos 0-14 del plan de notebook de predicción de demanda.

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## Paso 0 — Setup del entorno

In [ ]:
import sklearn
assert sklearn.__version__ is not None
print(f'OK - scikit-learn {sklearn.__version__} cargado correctamente')

## Paso 1 — Cargar y validar el dataset

In [ ]:
DATA_PATH = Path('..') / 'data' / 'processed' / 'dataset' / 'dataset_limpio.xlsx'
df = pd.read_excel(DATA_PATH)
print(df.shape)
print(df.dtypes)
df.head()

In [ ]:
required_cols = {'Fecha', 'Ataud_Modelo', 'Monto', 'Monto_winsorizado', 'Forma de pago', 'Capilla'}
assert required_cols.issubset(set(df.columns)), f'Faltan columnas: {required_cols - set(df.columns)}'
assert df.shape[0] > 0, 'El dataset está vacío'
assert df['Fecha'].dtype.kind == 'M' or pd.api.types.is_datetime64_any_dtype(df['Fecha']), 'Fecha no es datetime'
print('OK - dataset válido, filas:', df.shape[0])

## Paso 2 — Limpieza básica y periodo mensual

In [ ]:
df = df.copy()
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Periodo'] = df['Fecha'].dt.to_period('M').astype(str)
df = df.dropna(subset=['Fecha'])

assert df['Fecha'].isna().sum() == 0, 'Aún hay fechas nulas'
assert df['Periodo'].str.match(r'^\d{4}-\d{2}$').all(), 'Formato de Periodo inválido'
print('OK - periodos generados:', df['Periodo'].nunique())

## Paso 3 — Agrupar modelos de ataúd en categorías

In [ ]:
TOP_CATEGORIAS = ['Americano', 'Lincoln', 'Imperial', 'sin_ataud', 'Madera', 'Biblia', 'Principe']

def categorizar(modelo):
    if pd.isna(modelo): return 'Otros'
    for cat in TOP_CATEGORIAS:
        if cat.lower() in str(modelo).lower(): return cat
    return 'Otros'

df['Categoria_Ataud'] = df['Ataud_Modelo'].apply(categorizar)
print(df['Categoria_Ataud'].value_counts())

In [ ]:
assert df['Categoria_Ataud'].notna().all(), 'Hay categorías nulas'
n_cats = df['Categoria_Ataud'].nunique()
assert 3 <= n_cats <= 10, f'Número de categorías fuera de rango: {n_cats}'
counts = df['Categoria_Ataud'].value_counts()
assert (counts.drop('Otros', errors='ignore') >= 5).all(), 'Categoría con muy pocos datos'
print('OK - categorías:', n_cats)

## Paso 4 — Tabla de proporciones (categoría → modelo específico)

In [ ]:
conteo = df.groupby(['Categoria_Ataud', 'Ataud_Modelo']).size().reset_index(name='count')
proporciones = {}
for cat in conteo['Categoria_Ataud'].unique():
    sub = conteo[conteo['Categoria_Ataud'] == cat]
    total = sub['count'].sum()
    proporciones[cat] = {row['Ataud_Modelo']: row['count'] / total for _, row in sub.iterrows()}

for cat, dist in proporciones.items():
    total = sum(dist.values())
    assert abs(total - 1.0) < 1e-6, f'Proporciones de {cat} no suman 1'
print('OK - proporciones válidas para', len(proporciones), 'categorías')

## Paso 5 — Tabla de demanda mensual por categoría

In [ ]:
demanda = df.groupby(['Periodo', 'Categoria_Ataud']).size().reset_index(name='cantidad')

todos_periodos = pd.period_range(df['Fecha'].min(), df['Fecha'].max(), freq='M').astype(str)
todas_categorias = df['Categoria_Ataud'].unique()
idx_completo = pd.MultiIndex.from_product([todos_periodos, todas_categorias], names=['Periodo', 'Categoria_Ataud'])
demanda = demanda.set_index(['Periodo', 'Categoria_Ataud']).reindex(idx_completo, fill_value=0).reset_index()
demanda.head()

In [ ]:
assert demanda['cantidad'].isna().sum() == 0, 'Hay valores nulos'
assert (demanda['cantidad'] >= 0).all(), 'Cantidades negativas'
assert demanda.groupby('Categoria_Ataud').size().nunique() == 1, 'Series desbalanceadas'
print('OK - tabla de demanda:', demanda.shape)

## Paso 6 — Feature engineering

In [ ]:
demanda = demanda.sort_values(['Categoria_Ataud', 'Periodo']).reset_index(drop=True)
demanda['fecha_periodo'] = pd.to_datetime(demanda['Periodo'])
demanda['mes'] = demanda['fecha_periodo'].dt.month
demanda['anio'] = demanda['fecha_periodo'].dt.year
demanda['t'] = demanda.groupby('Categoria_Ataud').cumcount()

for lag in [1, 2, 3]:
    demanda[f'lag_{lag}'] = demanda.groupby('Categoria_Ataud')['cantidad'].shift(lag)

demanda['rolling_mean_3'] = demanda.groupby('Categoria_Ataud')['cantidad'].shift(1).rolling(3).mean()

demanda_model = pd.get_dummies(demanda, columns=['Categoria_Ataud'], prefix='cat')
demanda_model = demanda_model.dropna().reset_index(drop=True)
demanda_model.head()

In [ ]:
feature_cols = [c for c in demanda_model.columns if c.startswith('lag_') or c.startswith('cat_')]
assert len(feature_cols) > 0, 'No se generaron features'
assert demanda_model.isna().sum().sum() == 0, 'Quedan NaN'
assert demanda_model.shape[0] > 20, 'Muy pocas filas'
print('OK - dataset de modelado:', demanda_model.shape)

## Paso 7 — Split temporal (walk-forward)

In [ ]:
FEATURES = [c for c in demanda_model.columns if c not in ['Periodo', 'fecha_periodo', 'cantidad']]
TARGET = 'cantidad'

demanda_model = demanda_model.sort_values('fecha_periodo').reset_index(drop=True)
corte = int(len(demanda_model) * 0.8)
train = demanda_model.iloc[:corte]
test = demanda_model.iloc[corte:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

assert train['fecha_periodo'].max() <= test['fecha_periodo'].min(), 'Fuga temporal'
print(f'OK - train: {len(X_train)}, test: {len(X_test)}')

## Paso 8 — Baseline naive

In [ ]:
baseline_pred = X_test['lag_1']
mae_base = mean_absolute_error(y_test, baseline_pred)
rmse_base = root_mean_squared_error(y_test, baseline_pred)
print('Baseline MAE:', mae_base, 'RMSE:', rmse_base)
assert not np.isnan(mae_base), 'Baseline MAE inválido'
print('OK - baseline calculado')

## Paso 9 — Entrenar Random Forest

In [ ]:
modelo_rf = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=2, random_state=42)
modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)

assert hasattr(modelo_rf, 'estimators_'), 'El modelo no se entrenó'
assert len(pred_rf) == len(y_test), 'Longitud incorrecta'
assert (pred_rf >= 0).all(), 'Predicciones negativas'
print('OK - modelo entrenado')

## Paso 10 — Métricas vs baseline

In [ ]:
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = root_mean_squared_error(y_test, pred_rf)
r2_rf = r2_score(y_test, pred_rf)
mape_rf = np.mean(np.abs((y_test - pred_rf) / y_test.replace(0, np.nan))) * 100

metricas = {
    'baseline': {'MAE': float(mae_base), 'RMSE': float(rmse_base)},
    'random_forest': {'MAE': float(mae_rf), 'RMSE': float(rmse_rf), 'R2': float(r2_rf), 'MAPE': float(mape_rf)}
}
print(json.dumps(metricas, indent=2))
assert metricas['random_forest']['MAE'] <= metricas['baseline']['MAE'] * 1.15, 'Modelo no mejora al baseline'
print('OK - métricas validadas')

## Paso 11 — Precio promedio por categoría y monto esperado

In [ ]:
precio_promedio = df.groupby('Categoria_Ataud')['Monto_winsorizado'].mean().to_dict()

ultima_pred = test.assign(prediccion=pred_rf)
cat_cols = [c for c in FEATURES if c.startswith('cat_')]
ultima_pred['categoria'] = ultima_pred[cat_cols].idxmax(axis=1).str.replace('cat_', '')
ultima_pred['precio_promedio'] = ultima_pred['categoria'].map(precio_promedio)
ultima_pred['monto_esperado'] = ultima_pred['prediccion'] * ultima_pred['precio_promedio']
ultima_pred[['categoria', 'prediccion', 'precio_promedio', 'monto_esperado']].head(10)

In [ ]:
assert not ultima_pred['monto_esperado'].isna().any(), 'Montos nulos'
assert (ultima_pred['monto_esperado'] >= 0).all(), 'Monto negativo'
print('OK - monto esperado calculado')

## Paso 12 — Desglose a modelo específico

In [ ]:
def desglosar_por_modelo(categoria, cantidad_predicha, proporciones):
    dist = proporciones.get(categoria, {})
    return {modelo: round(cantidad_predicha * pct, 1) for modelo, pct in dist.items()}

ejemplo = desglosar_por_modelo('Lincoln', 12, proporciones)
print(ejemplo)
suma = sum(ejemplo.values())
assert abs(suma - 12) < 0.5, 'Desglose no suma'
print('OK - desglose funcional')

## Paso 13 — Alerta de reorden

In [ ]:
def alerta_reorden(stock_actual, demanda_predicha, umbral_seguridad=0.2):
    alertas = []
    for categoria, demanda in demanda_predicha.items():
        stock = stock_actual.get(categoria, 0)
        punto_reorden = demanda * (1 + umbral_seguridad)
        if stock < punto_reorden:
            alertas.append({
                'categoria': categoria,
                'stock_actual': stock,
                'demanda_predicha': round(float(demanda), 1),
                'unidades_a_comprar': round(float(punto_reorden - stock), 1)
            })
    return alertas

stock_ejemplo = {'Lincoln': 5, 'Americano': 10}
demanda_ejemplo = {'Lincoln': 12, 'Americano': 8}
resultado = alerta_reorden(stock_ejemplo, demanda_ejemplo)
print(resultado)

assert any(a['categoria'] == 'Lincoln' for a in resultado), 'No generó alerta para Lincoln'
assert not any(a['categoria'] == 'Americano' for a in resultado), 'Alerta incorrecta para Americano'
print('OK - alertas validadas')

## Paso 14 — Guardar artefactos

In [ ]:
OUTPUT_DIR = Path('..') / 'notebooks' / 'models'
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(modelo_rf, OUTPUT_DIR / 'modelo_demanda_rf.pkl')

cat_features = [c.replace('cat_', '') for c in FEATURES if c.startswith('cat_')]
metadata = {
    'features': FEATURES,
    'cat_features': cat_features,
    'top_categorias': TOP_CATEGORIAS,
    'proporciones_modelo_especifico': proporciones,
    'precio_promedio_categoria': {k: float(v) for k, v in precio_promedio.items()},
    'metricas': metricas,
    'fecha_entrenamiento': pd.Timestamp.now().isoformat(),
    'ultimo_periodo_entrenado': demanda_model['Periodo'].max()
}

with open(OUTPUT_DIR / 'demanda_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Archivos generados:', list(OUTPUT_DIR.glob('*')))

modelo_cargado = joblib.load(OUTPUT_DIR / 'modelo_demanda_rf.pkl')
pred_check = modelo_cargado.predict(X_test.iloc[:1])
assert len(pred_check) == 1
print('OK - artefactos guardados y verificados')

## Checklist final
- [x] Todas las aserciones pasaron
- [x] `modelo_demanda_rf.pkl` existe y se recarga correctamente
- [x] `demanda_metadata.json` contiene features, categorías, proporciones, precios, métricas
- [x] Métricas documentadas para la tesis